In [11]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, log_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer, make_column_selector
import os
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/CustomerChurn/')

In [12]:
customerchurn = pd.read_csv("train.csv",index_col='id')
customerchurn

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
id,,,,,,,,,,,,,,,,,,,,
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,Female,0,No,No,1,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No


In [13]:
customerchurn.info()

<class 'pandas.core.frame.DataFrame'>
Index: 594194 entries, 0 to 594193
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   gender            594194 non-null  object 
 1   SeniorCitizen     594194 non-null  int64  
 2   Partner           594194 non-null  object 
 3   Dependents        594194 non-null  object 
 4   tenure            594194 non-null  int64  
 5   PhoneService      594194 non-null  object 
 6   MultipleLines     594194 non-null  object 
 7   InternetService   594194 non-null  object 
 8   OnlineSecurity    594194 non-null  object 
 9   OnlineBackup      594194 non-null  object 
 10  DeviceProtection  594194 non-null  object 
 11  TechSupport       594194 non-null  object 
 12  StreamingTV       594194 non-null  object 
 13  StreamingMovies   594194 non-null  object 
 14  Contract          594194 non-null  object 
 15  PaperlessBilling  594194 non-null  object 
 16  PaymentMethod     594194 

In [14]:
customerchurn.isna().sum().sum()

0

In [15]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
transformer = ColumnTransformer(transformers=[('OHE',ohe,make_column_selector(dtype_include=object)),
                                            ]
                               ,remainder='passthrough',
                               verbose_feature_names_out=False
                               ).set_output(transform='pandas')

In [16]:
X, y = customerchurn.drop("Churn", axis = 1), customerchurn["Churn"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26,stratify=customerchurn['Churn'])


In [17]:
X_train_ohe = transformer.fit_transform(X_train)
X_test_ohe = transformer.transform(X_test)

In [18]:
scores=[]
k=[1,2,3,4,5,6,7,8,9,10]
for i in tqdm(k):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_ohe,y_train)
    y_pred = knn.predict_proba(X_test_ohe)
    score = log_loss(y_test,y_pred)
    scores.append([i,score])
df_scores = pd.DataFrame(scores,columns=['k','scores'])
df_scores.sort_values('scores',ascending=True)

100%|███████████████████████████████████████████| 10/10 [14:53<00:00, 89.40s/it]


,k,scores
9,10,0.750221
8,9,0.812187
7,8,0.892062
6,7,1.003066
5,6,1.150184
4,5,1.377602
3,4,1.723698
2,3,2.325421
1,2,3.573893
0,1,6.936612


In [19]:
ss = StandardScaler()

In [20]:
X_train_scaled = ss.fit_transform(X_train_ohe)
X_test_scaled = ss.transform(X_test_ohe)

In [21]:
scores=[]
k=[1,2,3,4,5,6,7,8,9,10]
for i in tqdm(k):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_scaled,y_train)
    y_pred = knn.predict_proba(X_test_scaled)
    score = log_loss(y_test,y_pred)
    scores.append([i,score])
df_scores = pd.DataFrame(scores,columns=['k','scores'])
df_scores.sort_values('scores',ascending=True)

100%|███████████████████████████████████████████| 10/10 [14:56<00:00, 89.63s/it]


,k,scores
9,10,0.726346
8,9,0.786900
7,8,0.867079
6,7,0.974037
5,6,1.130276
4,5,1.361931
3,4,1.717608
2,3,2.337280
1,2,3.579391
0,1,6.922863


### Inferencing

In [22]:
X_ohe = transformer.fit_transform(X)
X_scaled = ss.fit_transform(X_ohe)
bm = KNeighborsClassifier(n_neighbors = 10, n_jobs=-1)
bm.fit(X_scaled, y)

KNeighborsClassifier(n_jobs=-1, n_neighbors=10)

In [23]:
tst = pd.read_csv("test.csv",index_col='id')
tst_ohe = transformer.transform(tst)
tst_scaled = ss.transform(tst_ohe)
y_pred_proba = bm.predict_proba(tst_scaled)

In [24]:
sample = pd.read_csv("sample_submission.csv")
sample["Churn"] = y_pred_proba[:, 1]

In [25]:
sample

,id,Churn
0,594194,0.1
1,594195,0.0
2,594196,0.1
3,594197,0.0
4,594198,0.5
...,...,...
254650,848844,0.0
254651,848845,0.7
254652,848846,0.4
254653,848847,0.0


In [26]:
sample.to_csv("knn_submission.csv", index = False)